# Target Type & Intent Classification Pipeline

End-to-end inference notebook: loads a trajectory, runs the multi-model fusion stack, and saves results to `output/demo/output.csv`.

## 1. Imports

In [1]:
%load_ext autoreload
%autoreload 2

from src.config import *
from src.preprocess import *
from src.model_training.models import *
from src.postprocess import *
from src.optimize.optimization import *

import tracemalloc
import time

## 2. Load Parameters from `params.py`

In [2]:
import params

# All parameters (K1, K2, M, W, accuracies, paths, labels, etc.)
# are defined in params.py — edit there, not here.
print(f"K1={params.K1}  K2={params.K2}  M={params.M}  W={params.W}")
print(f"FFNN_ACCURACY={params.FFNN_ACCURACY}  LSTM_ACCURACY={params.LSTM_ACCURACY}  SVM_ACCURACY={params.SVM_ACCURACY}")
print(f"DT_ACCURACY={params.DT_ACCURACY}  OPTIMAL_X={params.OPTIMAL_X}")

K1=9  K2=4  M=1000  W=50
FFNN_ACCURACY=0.808062  LSTM_ACCURACY=0.747063  SVM_ACCURACY=0.7595
DT_ACCURACY=0.92825  OPTIMAL_X=0.7356230806


## 3. Load Data & Pre-trained Models

In [3]:
import joblib
import pandas as pd
import matplotlib
matplotlib.rcParams["font.family"] = "Serif"
from tensorflow.keras.models import load_model

# ── Raw trajectory ─────────────────────────────────────────────────────────
data = pd.read_csv(params.INPUT_PATH, header=None)

# ── Pre-processing scalers & encoder ───────────────────────────────────────
trans_standard = joblib.load(params.SCALER_TRANSFORM_PATH)
stat_standard  = joblib.load(params.SCALER_STAT_PATH)
encoding       = joblib.load(params.OHE_PATH)

# ── Classification models ──────────────────────────────────────────────────
ffnn_model          = load_model(params.FFNN_MODEL_PATH)
lstm_model          = load_model(params.LSTM_MODEL_PATH)
svm_model           = joblib.load(params.SVM_MODEL_PATH)
decision_tree_model = joblib.load(params.DT_MODEL_PATH)

# ── Split trajectory into type / intent feature sets ──────────────────────
type_data   = pd.concat([data.iloc[:, :9000],      data.iloc[:, -2]], axis=1)
intent_data = pd.concat([data.iloc[:, 9000:13000], data.iloc[:, -1]], axis=1)

print("Trajectory shape:", data.shape)
print("Type features:  ", type_data.shape, " | Intent features:", intent_data.shape)

Trajectory shape: (4000, 13002)
Type features:   (4000, 9001)  | Intent features: (4000, 4001)


## 4. Select Trajectory for Inference

In [4]:
i = int(input("Enter trajectory index for testing: "))

target_type   = pd.DataFrame([type_data.iloc[i,   :]], columns=type_data.columns)
target_intent = pd.DataFrame([intent_data.iloc[i, :]], columns=intent_data.columns)

# True target-type label for this trajectory (second-to-last column of raw data)
true_type_label = str(data.iloc[i, -2])

# Sanitise label for use in a filename (replace spaces / special chars with _)
import re
safe_label = re.sub(r"[^\w]+", "_", true_type_label).strip("_")

# Final output filename: ownship_vs_<label>_trajectory_<index>.csv
OUTPUT_PATH = os.path.join(
    params.DEMO_OUTPUT_DIR,
    f"ownship_vs_{safe_label}_trajectory_{i}.csv"
)

print(f"Selected trajectory : {i}")
print(f"True type label     : {true_type_label}")
print(f"Output will be saved: {OUTPUT_PATH}")

Enter trajectory index for testing:  1683


Selected trajectory : 1683
True type label     : CruiseMissile
Output will be saved: C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\output\demo\ownship_vs_CruiseMissile_trajectory_1683.csv


## 5. Preprocessing – Target Type Branch

In [5]:
# ── Sliding-window slicing ─────────────────────────────────────────────────
test_slices    = slice_dataset(target_type, params.K1, params.M, params.W)

# ── Feature extraction ─────────────────────────────────────────────────────
test_transform = transform_dataset(test_slices, params.K1, params.W)
test_stat      = SFE_dataset(test_slices, params.K1, params.W)

# ── Feature scaling ────────────────────────────────────────────────────────
test_stat_scaled  = stat_standard.transform(pd.DataFrame(test_stat).iloc[:, :-1])
test_trans_scaled = trans_standard.transform(pd.DataFrame(test_transform).iloc[:, :-1])

# ── Label encoding ─────────────────────────────────────────────────────────
y_test_encoded = encoding.transform(test_stat.iloc[:, -1].values.reshape(-1, 1))

print("Stat features shape:      ", test_stat_scaled.shape)
print("Transform features shape: ", test_trans_scaled.shape)

Stat features shape:       (20, 99)
Transform features shape:  (20, 450)


## 6. Target-Type Classification & Fusion

**Pipeline:** FFNN + LSTM + SVM → Belief mapping → Classifier fusion → Temporal fusion

In [6]:
tracemalloc.start()
start_time = time.time()

# ── FFNN ───────────────────────────────────────────────────────────────────
ffnn_prob  = ffnn_model.predict(test_stat_scaled, verbose=0)

# ── LSTM (requires 3-D input: samples × window × features) ────────────────
lstm_input = test_trans_scaled.reshape(test_trans_scaled.shape[0], params.W, params.K1)
lstm_prob  = lstm_model.predict(lstm_input, verbose=0)

# ── SVM ────────────────────────────────────────────────────────────────────
svm_prob   = svm_model.predict_proba(test_stat_scaled)

# ── Belief mapping ─────────────────────────────────────────────────────────
svm_belief, ffnn_belief, lstm_belief = belief_mapping_type(
    svm_prob,  params.SVM_ACCURACY,
    ffnn_prob, params.FFNN_ACCURACY,
    lstm_prob, params.LSTM_ACCURACY,
)

# ── Classifier fusion ──────────────────────────────────────────────────────
fused_ffnn_lstm     = fuse_ffnn_lstm(ffnn_belief, lstm_belief)
fused_ffnn_lstm_svm = fuse_ffnn_lstm_svm(fused_ffnn_lstm, svm_belief)

# ── Temporal fusion ────────────────────────────────────────────────────────
final_fused = temporal_fusion_type(fused_ffnn_lstm_svm)

# ── Level-1 output table ───────────────────────────────────────────────────
true_label    = pd.DataFrame(test_slices).iloc[:, -1]
level1_output = pd.concat(
    [pd.DataFrame(final_fused), true_label], axis=1
).reset_index(drop=True)

print("Level-1 output shape:", level1_output.shape)

Level-1 output shape: (20, 12)


## 7. Intent Classification & Temporal Fusion

In [7]:
# ── Membership function on level-1 output ─────────────────────────────────
Test_Membership = membership_function(labelling(level1_output))

# ── Intent feature engineering ─────────────────────────────────────────────
intent_slices = slice_dataset(target_intent, params.K2, params.M, params.W)
rule_test     = extract_rule(intent_slices, params.K2, params.W)
test_update   = Cond(
    pd.DataFrame(rule_test).iloc[:, :-1],
    pd.DataFrame(rule_test).iloc[:, -1],
)

# ── Map threat features ────────────────────────────────────────────────────
Intent_Featur  = map_threat(test_update, Test_Membership)
test_features  = Intent_Featur.drop(columns=["Target"], errors="ignore").to_numpy()

# ── Decision Tree ──────────────────────────────────────────────────────────
DT_prob = decision_tree_model.predict_proba(test_features)

# ── Belief mapping ─────────────────────────────────────────────────────────
BM_intent = belief_mapping_intent(DT_prob, params.DT_ACCURACY)

# ── Temporal fusion ────────────────────────────────────────────────────────
combined_mass_functions = temporal_fusion_intent(
    pd.DataFrame(BM_intent), params.OPTIMAL_X, batch_size=params.INTENT_BATCH_SIZE
)

end_time = time.time()
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Execution time (per window): {(end_time - start_time) / params.W:.4f} s")
print(f"Memory – current: {current/1024/1024:.2f} MB | peak: {peak/1024/1024:.2f} MB")

Execution time (per window): 0.2494 s
Memory – current: 3.34 MB | peak: 3.43 MB


## 8. Format Results

In [8]:
import numpy as np

def _add_header(df, row_labels, col_labels):
    """Prepend a time-step row and a label column to a transposed prob DataFrame."""
    df_t = df.T
    df_t = pd.concat([pd.DataFrame([row_labels]), df_t], axis=0, ignore_index=True)
    df_t = pd.concat([pd.DataFrame(col_labels),  df_t], axis=1, ignore_index=True)
    return df_t

# ── Table 1: Target-type probabilities ────────────────────────────────────
type_predicted = np.vectorize(params.TARGET_TYPE_LABELS.get)(
    np.argmax(pd.DataFrame(final_fused).values, axis=1)
)
prob_1_data = pd.concat(
    [pd.DataFrame(final_fused), true_label, pd.DataFrame(type_predicted)], axis=1
)
prob_1 = _add_header(prob_1_data, params.TIME_STEPS, params.TARGET_TYPE_COLS)

# ── Table 2: DT intent probabilities ──────────────────────────────────────
true_intent         = pd.DataFrame(test_features).iloc[:, -1]
intent_predicted_dt = np.vectorize(params.INTENT_LABELS.get)(
    np.argmax(pd.DataFrame(DT_prob).values, axis=1)
)
prob_2_data = pd.concat(
    [pd.DataFrame(DT_prob), true_intent, pd.DataFrame(intent_predicted_dt)], axis=1
)
prob_2 = _add_header(prob_2_data, params.TIME_STEPS, params.INTENT_DT_COLS)

# ── Table 3: Temporally fused intent probabilities ─────────────────────────
intent_predicted_fused = np.vectorize(params.INTENT_TEMPORAL_LABELS.get)(
    np.argmax(combined_mass_functions.values, axis=1)
)
true_intent_fused = Intent_Featur.iloc[:, -1]
prob_3_data = pd.concat(
    [combined_mass_functions, true_intent_fused, pd.DataFrame(intent_predicted_fused)], axis=1
)
prob_3 = _add_header(prob_3_data, params.TIME_STEPS, params.INTENT_FUSED_COLS)

print("Table shapes — Type:", prob_1.shape, "| DT Intent:", prob_2.shape, "| Fused Intent:", prob_3.shape)

Table shapes — Type: (14, 21) | DT Intent: (6, 21) | Fused Intent: (7, 21)


## 9. Save Output

In [9]:
_empty = lambda df: pd.DataFrame([[None] * df.shape[1]], columns=df.columns)

overall = pd.concat(
    [
        prob_1,
        _empty(prob_1),
        _empty(prob_1),
        prob_2.iloc[1:, :],   # skip duplicated header row
        _empty(prob_1),
        _empty(prob_1),
        prob_3.iloc[1:, :],
    ],
    axis=0,
    ignore_index=True,
)

# Round numeric values to 3 decimal places
overall = overall.applymap(lambda x: f"{x:.3f}" if isinstance(x, (int, float)) else x)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
overall.to_csv(OUTPUT_PATH, header=None, index=False)

print(f"Results saved → {OUTPUT_PATH}")
print(overall.head())

Results saved → C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\output\demo\ownship_vs_CruiseMissile_trajectory_1683.csv
                           0      1      2      3      4      5      6   \
0                                t_1    t_2    t_3    t_4    t_5    t_6   
1                      Bomber  0.000  0.000  0.000  0.000  0.000  0.000   
2  CivilianCommercialAirliner  0.000  0.000  0.000  0.000  0.000  0.000   
3          CivilianHelicopter  0.000  0.000  0.000  0.000  0.000  0.000   
4              zCruiseMissile  0.948  0.998  0.998  0.998  0.998  0.998   

      7      8      9   ...     11     12     13     14     15     16     17  \
0    t_7    t_8    t_9  ...   t_11   t_12   t_13   t_14   t_15   t_16   t_17   
1  0.000  0.000  0.000  ...  0.000  0.000  0.000  0.000  0.000  0.000  0.000   
2  0.000  0.000  0.000  ...  0.000  0.000  0.000  0.000  0.000  0.000  0.000   
3  0.000  0.000  0.000  ...  0.000  0.000  0.000  0.000  0.000  0.000  0.000   
4  0.998  0.99

# 10. Animated GIF Generation

This section generates animated GIFs for:

- **Target Type Belief Distribution**
- **Target Intent Belief Distribution**

The animations are automatically saved into:

`Output/Online_Plot/Gif/`


In [10]:
# -------------------------------- Animated GIF + PNG Generation -------------------------------- #

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import os

plt.rcParams["font.family"] = "Serif"
plt.style.use("seaborn-v0_8")

# ==========================================================================================
#                               CREATE OUTPUT DIRECTORIES
# ==========================================================================================

gif_output_dir = os.path.join(os.getcwd(), 'output', 'demo', 'gif_belief_distribution')
png_output_dir = os.path.join(os.getcwd(), 'output', 'demo', 'plot')

os.makedirs(gif_output_dir, exist_ok=True)
os.makedirs(png_output_dir, exist_ok=True)

# ==========================================================================================
#                               TARGET TYPE GIF
# ==========================================================================================

type_df = prob_1.iloc[1:, :]
type_df = type_df.iloc[0:11, :]

x_values = pd.to_numeric(type_df.columns[1:], errors='coerce')

fig, ax = plt.subplots(figsize=(24, 12))

colors = plt.cm.tab10(np.linspace(0, 1, len(type_df)))
line_styles = ['-', '--', '-.', ':']

type_labels = [
    "Bomber",
    "Civilian commercial airliner",
    "Civilian helicopter",
    "Air-to-air missile",
    "Military helicopter",
    "Multirole fighter",
    "Small civilian aircraft",
    "Small unmanned aircraft",
    r"$\theta_1$",
    r"$\theta_2$",
    r"$\Omega$"
]

lines = []

for idx, label in enumerate(type_labels):

    line, = ax.plot(
        [],
        [],
        label=label,
        color=colors[idx],
        linestyle=line_styles[idx % len(line_styles)],
        linewidth=4
    )

    lines.append(line)

# Axis labels
ax.set_xlabel("Time (sec)", fontsize=22)
ax.set_ylabel("Belief Value", fontsize=22)

# Tick size
ax.tick_params(axis='both', which='major', labelsize=18)

# Convert numeric
type_numeric = type_df.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

type_numeric = type_numeric.fillna(1e-10)
type_numeric[type_numeric == 0] = 1e-10

y_max = type_numeric.max().max()

# Axis limits
ax.set_xlim(x_values.min(), x_values.max())
ax.set_ylim(0, y_max + 0.1)

# Border styling
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(1.5)

ax.spines[['top', 'right']].set_visible(False)

# GRID ON
ax.grid(True, linestyle='--', linewidth=1.0, alpha=0.8)

# Background
ax.set_facecolor('white')

# Legend
ax.legend(
    title="Target Type",
    loc='upper left',
    bbox_to_anchor=(1.02, 1),
    fontsize=16,
    title_fontsize=18,
    frameon=True,
    shadow=True
)

# Title
title = ax.set_title(
    "Target Type Belief Distribution",
    fontsize=24,
    fontweight='bold'
)

# Update function
def update_type(frame):

    for i, line in enumerate(lines):

        y_data = pd.to_numeric(
            type_df.iloc[i, 1:frame+2],
            errors='coerce'
        )

        line.set_data(
            x_values[:frame+1],
            y_data
        )

    current_time = x_values[frame]

    title.set_text(
        f"Target Type Belief Distribution | Time = {current_time:.1f} sec"
    )

    return lines + [title]

# Animation
type_anim = animation.FuncAnimation(
    fig,
    update_type,
    frames=len(x_values),
    interval=500,
    blit=False,
    repeat=True
)

# GIF SAVE PATH
type_gif_path = os.path.join(
    gif_output_dir,
    f"{safe_label}_trajectory_{i}_type.gif"
)

# SAVE GIF
type_anim.save(
    type_gif_path,
    writer='pillow',
    fps=2
)

# PNG SAVE PATH
type_png_path = os.path.join(
    png_output_dir,
    f"{safe_label}_trajectory_{i}_type.png"
)

# FINAL FRAME UPDATE BEFORE PNG SAVE
update_type(len(x_values) - 1)

# SAVE PNG
fig.savefig(
    type_png_path,
    bbox_inches='tight',
    dpi=300
)

plt.close(fig)

print(f"Target Type GIF saved successfully at:\n{type_gif_path}")
print(f"Target Type PNG saved successfully at:\n{type_png_path}")

# ==========================================================================================
#                               TARGET INTENT GIF
# ==========================================================================================

intent_df = prob_3.iloc[1:, :]
intent_df = intent_df.iloc[0:4, :]

x_values = pd.to_numeric(intent_df.columns[1:], errors='coerce')

fig, ax = plt.subplots(figsize=(24, 12))

colors = plt.cm.tab10(np.linspace(0, 1, len(intent_df)))
line_styles = ['-', '--', '-.', ':']

intent_labels = [
    "Hostile",
    "Non-Hostile",
    "Suspicious",
    r"$\Omega$"
]

intent_lines = []

for idx, label in enumerate(intent_labels):

    line, = ax.plot(
        [],
        [],
        label=label,
        color=colors[idx],
        linestyle=line_styles[idx % len(line_styles)],
        linewidth=4
    )

    intent_lines.append(line)

# Axis labels
ax.set_xlabel("Time (sec)", fontsize=22)
ax.set_ylabel("Belief Value", fontsize=22)

# Tick size
ax.tick_params(axis='both', which='major', labelsize=18)

# Convert numeric
intent_numeric = intent_df.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

intent_numeric = intent_numeric.fillna(1e-10)
intent_numeric[intent_numeric == 0] = 1e-10

y_max = intent_numeric.max().max()

# Axis limits
ax.set_xlim(x_values.min(), x_values.max())
ax.set_ylim(0, y_max + 0.1)

# Border styling
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(1.5)

ax.spines[['top', 'right']].set_visible(False)

# GRID ON
ax.grid(True, linestyle='--', linewidth=1.0, alpha=0.8)

# Background
ax.set_facecolor('white')

# Legend
ax.legend(
    title="Target Intent",
    loc='upper left',
    bbox_to_anchor=(1.02, 1),
    fontsize=16,
    title_fontsize=18,
    frameon=True,
    shadow=True
)

# Title
intent_title = ax.set_title(
    "Target Intent Belief Distribution",
    fontsize=24,
    fontweight='bold'
)

# Update function
def update_intent(frame):

    for i, line in enumerate(intent_lines):

        y_data = pd.to_numeric(
            intent_df.iloc[i, 1:frame+2],
            errors='coerce'
        )

        line.set_data(
            x_values[:frame+1],
            y_data
        )

    current_time = x_values[frame]

    intent_title.set_text(
        f"Target Intent Belief Distribution | Time = {current_time:.1f} sec"
    )

    return intent_lines + [intent_title]

# Animation
intent_anim = animation.FuncAnimation(
    fig,
    update_intent,
    frames=len(x_values),
    interval=500,
    blit=False,
    repeat=True
)

# GIF SAVE PATH
intent_gif_path = os.path.join(
    gif_output_dir,
    f"{safe_label}_trajectory_{i}_intent.gif"
)

# SAVE GIF
intent_anim.save(
    intent_gif_path,
    writer='pillow',
    fps=2
)

# PNG SAVE PATH
intent_png_path = os.path.join(
    png_output_dir,
    f"{safe_label}_trajectory_{i}_intent.png"
)

# FINAL FRAME UPDATE BEFORE PNG SAVE
update_intent(len(x_values) - 1)

# SAVE PNG
fig.savefig(
    intent_png_path,
    bbox_inches='tight',
    dpi=300
)

plt.close(fig)

print(f"Target Intent GIF saved successfully at:\n{intent_gif_path}")
print(f"Target Intent PNG saved successfully at:\n{intent_png_path}")

Target Type GIF saved successfully at:
C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\output\demo\gif_belief_distribution\CruiseMissile_trajectory_1683_type.gif
Target Type PNG saved successfully at:
C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\output\demo\plot\CruiseMissile_trajectory_1683_type.png
Target Intent GIF saved successfully at:
C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\output\demo\gif_belief_distribution\CruiseMissile_trajectory_1683_intent.gif
Target Intent PNG saved successfully at:
C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\output\demo\plot\CruiseMissile_trajectory_1683_intent.png


In [25]:
# ==========================================================================================
#                   3D TRAJECTORY — INTERACTIVE PLOTLY (HTML OUTPUT)
# ==========================================================================================
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
i = 2690
# ==========================================================================================
#                               CREATE OUTPUT DIRECTORY
# ==========================================================================================
html_output_dir = os.path.join(os.getcwd(), 'output', 'demo', 'plotly_trajectory')
os.makedirs(html_output_dir, exist_ok=True)
data = pd.read_csv(os.path.join(os.getcwd(), 'input', 'trajectory_1.csv'), header=None)
# ==========================================================================================
#                               TARGET TRAJECTORY
# ==========================================================================================
T_x_axis = pd.to_numeric(data.iloc[i, 0:1000],    errors='coerce').reset_index(drop=True)
T_y_axis = pd.to_numeric(data.iloc[i, 1000:2000], errors='coerce').reset_index(drop=True)
T_z_axis = pd.to_numeric(data.iloc[i, 2000:3000], errors='coerce').reset_index(drop=True)

# ==========================================================================================
#                               OWN-SHIP TRAJECTORY
# ==========================================================================================
ows = pd.read_csv(
    os.path.join(os.getcwd(), 'Ownship_traj', 'MultiroleFighter.csv'),
    header=None
)
O_x_axis = pd.to_numeric(ows.iloc[0, :], errors='coerce').reset_index(drop=True)
O_y_axis = pd.to_numeric(ows.iloc[1, :], errors='coerce').reset_index(drop=True)
O_z_axis = pd.to_numeric(ows.iloc[2, :], errors='coerce').reset_index(drop=True)

# ==========================================================================================
#                               ANIMATION FRAMES
#   Each frame grows the trail from step 0 up to step k.
#   Capped at 100 keyframes so the HTML file stays a reasonable size.
# ==========================================================================================
num_frames    = min(len(T_x_axis), len(O_x_axis))
step          = max(1, num_frames // 100)
frame_indices = list(range(0, num_frames, step))
if frame_indices[-1] != num_frames - 1:
    frame_indices.append(num_frames - 1)

frames = []
for k in frame_indices:
    frames.append(go.Frame(
        name=str(k),
        data=[
            # Target trail
            go.Scatter3d(
                x=T_x_axis.iloc[:k+1], y=T_y_axis.iloc[:k+1], z=T_z_axis.iloc[:k+1],
                mode='lines', line=dict(color='crimson', width=3),
                name=safe_label, showlegend=False,
            ),
            # Target head marker
            go.Scatter3d(
                x=[T_x_axis.iloc[k]], y=[T_y_axis.iloc[k]], z=[T_z_axis.iloc[k]],
                mode='markers', marker=dict(color='crimson', size=8),
                showlegend=False,
            ),
            # Own-ship trail
            go.Scatter3d(
                x=O_x_axis.iloc[:k+1], y=O_y_axis.iloc[:k+1], z=O_z_axis.iloc[:k+1],
                mode='lines', line=dict(color='royalblue', width=3),
                name='Own-ship', showlegend=False,
            ),
            # Own-ship head marker
            go.Scatter3d(
                x=[O_x_axis.iloc[k]], y=[O_y_axis.iloc[k]], z=[O_z_axis.iloc[k]],
                mode='markers', marker=dict(color='royalblue', size=8),
                showlegend=False,
            ),
        ],
        layout=go.Layout(
            title_text=f'3D Trajectory Visualization &nbsp;|&nbsp; Time Step = {k}'
        ),
    ))

# ==========================================================================================
#                               INITIAL TRACES  (frame 0 + legend entries)
# ==========================================================================================
initial_traces = [
    go.Scatter3d(
        x=T_x_axis.iloc[:1], y=T_y_axis.iloc[:1], z=T_z_axis.iloc[:1],
        mode='lines', line=dict(color='crimson', width=3),
        name=safe_label, showlegend=True,
    ),
    go.Scatter3d(
        x=[T_x_axis.iloc[0]], y=[T_y_axis.iloc[0]], z=[T_z_axis.iloc[0]],
        mode='markers', marker=dict(color='crimson', size=8),
        showlegend=False,
    ),
    go.Scatter3d(
        x=O_x_axis.iloc[:1], y=O_y_axis.iloc[:1], z=O_z_axis.iloc[:1],
        mode='lines', line=dict(color='royalblue', width=3),
        name='Own-ship', showlegend=True,
    ),
    go.Scatter3d(
        x=[O_x_axis.iloc[0]], y=[O_y_axis.iloc[0]], z=[O_z_axis.iloc[0]],
        mode='markers', marker=dict(color='royalblue', size=8),
        showlegend=False,
    ),
]

# ==========================================================================================
#                               LAYOUT
#   aspectmode='auto' -> each axis keeps its own natural range (no suppression).
#   Switch to 'cube'  -> equal cubic box (matches old matplotlib set_box_aspect).
#   Switch to 'data'  -> axes scaled to true data proportions.
# ==========================================================================================
layout = go.Layout(
    title=dict(
        text='3D Trajectory Visualization',
        font=dict(family='Georgia, serif', size=20, color='black'),
        x=0.5, xanchor='center',
    ),
    scene=dict(
        aspectmode='auto',
        xaxis=dict(
            title='X-axis',
            backgroundcolor='rgb(245,245,245)',
            gridcolor='lightgrey',
            showbackground=True,
            tickfont=dict(size=10),
        ),
        yaxis=dict(
            title='Y-axis',
            backgroundcolor='rgb(245,245,245)',
            gridcolor='lightgrey',
            showbackground=True,
            tickfont=dict(size=10),
        ),
        zaxis=dict(
            title='Z-axis',
            backgroundcolor='rgb(235,235,245)',
            gridcolor='lightgrey',
            showbackground=True,
            tickfont=dict(size=10),
        ),
        camera=dict(eye=dict(x=1.6, y=1.6, z=0.8)),
    ),
    legend=dict(
        x=0.78, y=0.95,
        bgcolor='white', bordercolor='black', borderwidth=1.5,
        font=dict(size=13),
    ),
    paper_bgcolor='white',
    width=1100, height=750,
    # -- Play / Pause buttons --------------------------------------------------
    updatemenus=[dict(
        type='buttons', showactive=False, y=0.02, x=0.18, xanchor='right',
        buttons=[
            dict(
                label='Play', method='animate',
                args=[None, dict(frame=dict(duration=200, redraw=True),
                                 fromcurrent=True, transition=dict(duration=0))],
            ),
            dict(
                label='Pause', method='animate',
                args=[[None], dict(frame=dict(duration=0, redraw=False),
                                   mode='immediate', transition=dict(duration=0))],
            ),
        ],
    )],
    # -- Time-step slider ------------------------------------------------------
    sliders=[dict(
        steps=[
            dict(
                args=[[str(k)], dict(frame=dict(duration=200, redraw=True),
                                     mode='immediate', transition=dict(duration=0))],
                label=str(k), method='animate',
            )
            for k in frame_indices
        ],
        active=0, y=0, x=0.2, len=0.80,
        currentvalue=dict(prefix='Time Step: ', font=dict(size=13), xanchor='center'),
        transition=dict(duration=0),
    )],
)

# ==========================================================================================
#                               BUILD FIGURE & SAVE HTML
# ==========================================================================================
fig = go.Figure(data=initial_traces, layout=layout, frames=frames)

html_path = os.path.join(
    html_output_dir,
    f'{safe_label}_trajectory_{i}_3D_interactive.html'
)

fig.write_html(
    html_path,
    include_plotlyjs='cdn',
    full_html=True,
    auto_play=False,
)

# ==========================================================================================
#                               SUCCESS MESSAGE
# ==========================================================================================
print(
    f'Interactive 3D Trajectory HTML saved successfully at:\n'
    f'{html_path}'
)
print('Open the file in any browser — rotate, zoom, and pan freely in 3-D.')


Interactive 3D Trajectory HTML saved successfully at:
C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\output\demo\plotly_trajectory\CruiseMissile_trajectory_2690_3D_interactive.html
Open the file in any browser — rotate, zoom, and pan freely in 3-D.


In [18]:
i


1683